In [1]:
import wrds
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load your Kelly et al. dataset
kelly = pd.read_csv('datashare\\datashare.csv', low_memory=True)

# Understand what you have
print("Kelly dataset shape:", kelly.shape)
print("Date range:", kelly['DATE'].min(), "to", kelly['DATE'].max())
print("Unique permnos:", kelly['permno'].nunique())
print("Columns:", kelly.columns.tolist())


# Extract the unique permnos — this is your universe
# Every data pull will be filtered to ONLY these permnos
# No point pulling the entire CRSP universe
permnos = kelly['permno'].unique().tolist()
print(f"\nTotal unique permnos to pull: {len(permnos)}")

# Save permnos for reference
pd.Series(permnos).to_csv('kelly_permnos.csv', index=False)

Kelly dataset shape: (4117300, 97)
Date range: 19570131 to 20211231
Unique permnos: 32793
Columns: ['permno', 'DATE', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret', 'retvol',

In [ ]:
#conn = wrds.Connection(wrds_username='muhab')

# Verify access to what you need
print("Checking access...")
for lib, table in [('crsp', 'msf'), 
                   ('crsp', 'msenames'), 
                   ('comp', 'funda')]:
    try:
        test = conn.raw_sql(f"SELECT * FROM {lib}.{table} LIMIT 1")
        print(f"  {lib}.{table}: OK")
    except:
        print(f"  {lib}.{table}: NO ACCESS")

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Checking access...
  crsp.msf: OK
  crsp.msenames: OK
  comp.funda: OK


In [ ]:
"""
WHY THIS APPROACH:
We filter by permno from the start using SQL.
This is much faster than pulling all of CRSP and filtering afterwards.
CRSP has ~100M rows. Our permnos subset is 32793 rows.

ABOUT ret n+1:
In the Kelly dataset, the ret column on row (permno, date=2020-01) 
is the return EARNED in February 2020 — it is the NEXT month's return.
This is the prediction target.

When we pull from CRSP, ret on date 2020-02 is the return 
earned DURING February 2020.
So to match Kelly's convention:
    Kelly row date=2020-01 ret  ==  CRSP row date=2020-02 ret

We handle this by shifting ret back by one month after pulling.
This is the single most important thing to get right.

ABOUT THE DATE RANGE:
- Kelly dataset ends 2021-12-31
- We need returns for those rows, meaning we need CRSP up to 
  one month after: 2022-01-31 (to get the December 2021 ret)
- For the extension to Feb 2026, we need CRSP up to March 2026
  (to get the February 2026 ret)
- So we pull CRSP from 1957 to 2026-03-31 in one go
"""

def pull_crsp_returns(conn, permnos, 
                       start='1957-01-01', 
                       end='2026-03-31'):
    
    chunk_size = 500  # Reduced chunk size to avoid query timeouts
    chunks = [permnos[i:i+chunk_size] 
              for i in range(0, len(permnos), chunk_size)]
    
    all_chunks = []
    
    for i, chunk in enumerate(chunks):
        print(f"  Pulling chunk {i+1}/{len(chunks)}...")
        permno_str = ','.join(map(str, chunk))
        
        chunk_df = conn.raw_sql(f"""
            SELECT
                a.permno,
                a.date,
                a.ret,
                a.retx,
                a.prc,
                a.shrout,
                a.vol,
                b.shrcd,
                b.exchcd,
                b.siccd,
                c.dlret,
                c.dlstcd
            FROM crsp.msf AS a
            LEFT JOIN crsp.msenames AS b
                ON a.permno = b.permno
                AND b.namedt <= a.date
                AND a.date <= b.nameendt
            LEFT JOIN crsp.msedelist AS c
                ON a.permno = c.permno
                AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
            WHERE a.date BETWEEN '{start}' AND '{end}'
                AND a.permno IN ({permno_str})
        """, date_cols=['date'])
        
        all_chunks.append(chunk_df)
        
    crsp = pd.concat(all_chunks, ignore_index=True)
    print(f"Total rows: {len(crsp):,}")
    print(f"Unique permnos: {crsp['permno'].nunique():,}")
    
    return crsp


def clean_crsp_returns(crsp):
    """
    Cleans raw CRSP data and applies the n+1 shift
    to match Kelly et al. convention.
    """
    
    crsp = crsp.copy()
    crsp = crsp.sort_values(['permno', 'date'])
    
    # --- Delisting return adjustment (Shumway 1997) ---
    # Performance delistings (codes 500-599) with missing dlret
    # get assigned -30% — standard in the literature
    crsp['dlret'] = crsp['dlret'].fillna(0)
    perf_delist = crsp['dlstcd'].between(500, 599)
    crsp.loc[perf_delist & (crsp['dlret'] == 0), 'dlret'] = -0.30
    
    # Combine monthly return with delisting return
    crsp['ret_raw'] = crsp['ret'].copy()
    crsp['ret'] = (
        (1 + crsp['ret'].fillna(0)) * 
        (1 + crsp['dlret']) - 1
    )
    # Restore genuine missing (not just zero)
    crsp.loc[crsp['ret_raw'].isna() & 
             (crsp['dlret'] == 0), 'ret'] = np.nan
    
    # Market cap
    crsp['me'] = crsp['prc'].abs() * crsp['shrout'] / 1000
    
    # --- Apply the n+1 shift ---
    # Kelly's ret on date T = CRSP's ret on date T+1
    # So we shift ret BACK by one month within each permno
    # After shift: the ret on row (permno, date=Jan) is the 
    # return earned IN January, which is what Kelly calls 
    # the return FOR December (the previous month's row)
    crsp['ret'] = crsp.groupby('permno')['ret'].shift(-1)
    
    # Standardize date format to match Kelly: YYYY-MM-DD
    # Kelly uses the first day of month, CRSP uses last day
    # Align to month start to match Kelly's DATE format
    crsp['date'] = pd.to_datetime(crsp['date'])
    crsp['date'] = crsp['date'].dt.to_period('M').dt.to_timestamp()
    # MS = month start, so 2020-01-31 becomes 2020-01-01
    
    # Drop the last month per permno (ret is NaN after shift)
    crsp = crsp.dropna(subset=['ret'])
    
    # Final sort
    crsp = crsp.sort_values(['permno', 'date']).reset_index(drop=True)
    
    # Sanity checks
    print("After cleaning:")
    print(f"  Rows: {len(crsp):,}")
    print(f"  Return range: {crsp['ret'].min():.4f} to {crsp['ret'].max():.4f}")
    print(f"  Date range: {crsp['date'].min()} to {crsp['date'].max()}")
    
    return crsp


# Run it
crsp_raw = pull_crsp_returns(conn, permnos)


In [7]:
crsp_clean = clean_crsp_returns(crsp_raw)

# Save
crsp_clean.to_csv('crsp_returns.csv', index=False)
print("Saved: crsp_returns.csv")

After cleaning:
  Rows: 4,439,668
  Return range: -1.0000 to 39.0000
  Date range: 1957-01-01 00:00:00 to 2024-11-01 00:00:00
Saved: crsp_returns.csv


In [8]:
crsp_clean = pd.read_csv('crsp_returns.csv', parse_dates=['date'])

print("Date range:", crsp_clean['date'].min(), "to", crsp_clean['date'].max())
print("Rows:", len(crsp_clean))

print("\nReturn percentiles:")
print(crsp_clean['ret'].quantile([0.001, 0.01, 0.99, 0.999, 0.9999]))

print("\nTop 10 highest returns:")
print(crsp_clean.nlargest(10, 'ret')[['permno','date','ret','prc','shrout']])

Date range: 1957-01-01 00:00:00 to 2024-11-01 00:00:00
Rows: 4439668

Return percentiles:
0.0010   -0.685714
0.0100   -0.392182
0.9900    0.551546
0.9990    1.421053
0.9999    3.380952
Name: ret, dtype: float64

Top 10 highest returns:
         permno       date        ret       prc   shrout
940261    22298 2024-09-01  39.000000   1.17000   4524.0
49576     10349 1983-08-01  35.333332 -15.00000   1138.0
543832    15489 2024-04-01  26.583828   0.20340  53156.0
2124986   58748 1991-12-01  24.000000  -0.04688  16184.0
599019    16400 2018-12-01  19.883589  14.26000  27253.0
785923    19549 2024-11-01  19.486032   1.79000   2099.0
1958505   53154 1993-07-01  19.000000   0.03125   1973.0
834458    20412 2024-10-01  17.916666   3.00000   1332.0
1798310   48072 2020-12-01  17.604650   3.44000   7447.0
3974407   89301 2020-12-01  16.250530  18.84000  69747.0


In [9]:
# --- Inspect the problem more clearly ---
print("Price distribution of extreme return stocks:")
extreme = crsp_clean[crsp_clean['ret'] > 2.0]
print(f"Rows with ret > 200%: {len(extreme)}")
print(extreme['prc'].abs().describe())

# --- Apply two filters ---

# Filter 1: Price filter
# Exclude stocks trading below $1 at the START of the month
# These are penny stocks — untradeable at scale and noisy
# Kelly et al. and most systematic strategies do this
crsp_clean['prc_abs'] = crsp_clean['prc'].abs()
low_price_mask = crsp_clean['prc_abs'] < 1.0

print(f"\nRows removed by price filter (prc < $1): {low_price_mask.sum():,}")
print(f"As % of total: {low_price_mask.mean():.2%}")

crsp_filtered = crsp_clean[~low_price_mask].copy()

# Filter 2: Winsorize returns at 1st/99th percentile cross-sectionally
# This is exactly what Kelly et al. do
# It doesn't remove rows, just caps extreme values
p01 = crsp_filtered.groupby('date')['ret'].transform(
    lambda x: x.quantile(0.01)
)
p99 = crsp_filtered.groupby('date')['ret'].transform(
    lambda x: x.quantile(0.99)
)

crsp_filtered['ret_raw'] = crsp_filtered['ret'].copy()
crsp_filtered['ret'] = crsp_filtered['ret'].clip(lower=p01, upper=p99)

# --- Check result ---
print("\nAfter filtering:")
print(f"  Rows: {len(crsp_filtered):,}")
print(f"  Removed: {len(crsp_clean) - len(crsp_filtered):,}")
print(f"  Return range: {crsp_filtered['ret'].min():.4f} to {crsp_filtered['ret'].max():.4f}")
print(f"\nReturn percentiles after:")
print(crsp_filtered['ret'].quantile([0.001, 0.01, 0.99, 0.999]))

Price distribution of extreme return stocks:
Rows with ret > 200%: 1755
count    1744.000000
mean        2.317902
std         5.737598
min         0.015630
25%         0.312500
50%         0.750000
75%         1.940000
max        81.625000
Name: prc, dtype: float64

Rows removed by price filter (prc < $1): 209,858
As % of total: 4.73%

After filtering:
  Rows: 4,229,810
  Removed: 209,858
  Return range: -0.7248 to 1.7232

Return percentiles after:
0.001   -0.500000
0.010   -0.333333
0.990    0.437500
0.999    0.750000
Name: ret, dtype: float64


In [10]:
# --- Save ---
crsp_filtered.to_csv('crsp_returns.csv', index=False)
print("\nSaved: crsp_returns.csv")


Saved: crsp_returns.csv


In [11]:
# Check alignment with Kelly dataset

kelly['date'] = pd.to_datetime(kelly['DATE'].astype(str), format='%Y%m%d')
kelly['date'] = kelly['date'].dt.to_period('M').dt.to_timestamp()

test_permnos = kelly['permno'].sample(3, random_state=42).values
print("\nVerifying alignment with Kelly dataset...")

for p in test_permnos:
    k = (kelly[kelly['permno'] == p][['date','ret']]
         .sort_values('date')
         .head(8)
         .rename(columns={'ret': 'kelly_ret'}))
    
    c = (crsp_filtered[crsp_filtered['permno'] == p][['date','ret']]
         .sort_values('date')
         .head(8)
         .rename(columns={'ret': 'crsp_ret'}))
    
    comp = k.merge(c, on='date', how='inner')
    
    if len(comp) > 0:
        corr = comp['kelly_ret'].corr(comp['crsp_ret'])
        print(f"\npermno {p}: {len(comp)} matched rows, correlation = {corr:.6f}")
        print(comp.head(5).to_string(index=False))


Verifying alignment with Kelly dataset...


KeyError: "['ret'] not in index"